# GBM Data Leakage Correction

## 1. Objective

The objective of this notebook is to investigate and correct data leakage in the Gradient Boosting Model (GBM) used for energy consumption forecasting.

The original leakage issue was identified in T37, where some input features were found to contain information that would not be available when making a real day-ahead forecast.

This notebook extends the original T37 leakage investigation to all ten energy consumption series covered by the original GBM.

## 2. Problem Description

The original GBM uses some features that may introduce data leakage:

- **`roll_mean_24`**: the rolling average includes the energy consumption of the hour being predicted.
- **Indoor sensor variables**: temperature, occupancy and luminosity measurements may not be available 24 hours before the forecast.
- **Short-term weather and sensor information**: some variables must be shifted or replaced with information available at prediction time.

These features can make model performance appear better than it would be in a real forecasting scenario.

## 3. Methodology

The analysis will follow five steps:

1. Reproduce the original GBM configuration.
2. Identify features that use unavailable future information.
3. Train a corrected GBM using only information available at forecast time.
4. Compare the original and corrected models on the same evaluation periods.
5. Save the corrected predictions for subsequent Hybrid and final model-selection audits.

The original project files will remain unchanged.

## 4. Expected Outputs

- Comparison of original and corrected GBM performance.
- Updated GBM predictions.
- Analysis of performance changes across energy series.
- A reproducible notebook documenting the correction.

**Original references:** T37 – `T37_check_gbm_leakage.py` and `GBM_forallzones.ipynb`.

**Related work:** `01_GBM_Timezone_Audit.ipynb`, which addresses timestamp alignment separately.

In [ ]:

from google.colab import files
from pathlib import Path
import zipfile
import pandas as pd
import numpy as np

uploaded = files.upload()


Saving 2026G3-main.zip to 2026G3-main (1).zip


In [ ]:

zip_name = next(
    name for name in uploaded
    if name.lower().endswith(".zip")
)

extract_dir = Path("/content/tbda_leakage")

with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall(extract_dir)

project_dir = extract_dir / "2026G3-main"

print("Project folder:", project_dir.exists())
print("T37 script:", (
    project_dir / "cleaning_and_eda/T37_check_gbm_leakage.py"
).exists())
print("Original GBM notebook:", (
    project_dir / "models/GBM_forallzones.ipynb"
).exists())
print("T32 dataset:", (
    project_dir / "data/T32_cedint_f1_model_ready.csv.gz"
).exists())


Project folder: True
T37 script: True
Original GBM notebook: True
T32 dataset: True


In [ ]:

import json

notebook_path = project_dir / "models/GBM_forallzones.ipynb"

with open(notebook_path, encoding="utf-8") as f:
    original_gbm = json.load(f)

keywords = [
    "GradientBoostingRegressor",
    "roll_mean_24",
    "energy_lag_24",
    "features",
    "include_train",
    "include_eval"
]

for i, cell in enumerate(original_gbm["cells"]):
    source = "".join(cell.get("source", []))

    if any(word in source for word in keywords):
        print(f"\n{'='*15} CELL {i} {'='*15}")
        print(source[:4000])



=============== CELL 1 ===============
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt

# 1. Laad en bereid de dataset voor
df = pd.read_csv('cedint_f1_model_ready.csv')
df['timestamp_local'] = pd.to_datetime(df['timestamp_local'])

def calculate_ashrae_metrics(y_true, y_pred, split_name, zone, load):
    if len(y_true) == 0: return None
    
    n = len(y_true)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = root_mean_squared_error(y_true, y_pred)
    mean_y = np.mean(y_true)
    
    # ASHRAE 14 Formules
    cv_rmse = (rmse / mean_y) * 100 if mean_y != 0 else 0
    nmbe = (np.sum(y_true - y_pred) / (n * mean_y)) * 100 if mean_y != 0 else 0
    
    # Strenge limieten: CV(RMSE) <= 30% en NMBE tussen -10% en 10%
    ashrae_pass = (cv_rmse <= 30) and (-10 <= nmbe <= 10)
    
    return {
        'zone': zone, 'load': load, 'mode

In [ ]:

import pandas as pd

data_path = project_dir / "data/T32_cedint_f1_model_ready.csv.gz"

df = pd.read_csv(data_path, low_memory=False)

# Stessi criteri di training e valutazione del GBM originale
summary = (
    df.groupby(["zone", "load_type"])
    .agg(
        total_rows=("zone", "size"),
        train_rows=("include_train", "sum"),
        eval_rows=("include_eval", "sum")
    )
    .reset_index()
)

display(summary.sort_values(["zone", "load_type"]))

print("Total series:", len(summary))


,zone,load_type,total_rows,train_rows,eval_rows
0,NE,HVAC,26830,15169,6742
1,NE,Plugs & Sockets,26830,18084,6740
2,NW,Emerg. Lighting,26830,17175,6101
3,NW,HVAC,26830,16755,6180
4,NW,Hand Dryer,26830,0,0
5,NW,Lighting,26830,16734,6740
6,NW,Plugs & Sockets,26830,17589,6741
7,NW,RACK,26830,18161,6731
8,SE,Emerg. Lighting,26817,13571,5984
9,SE,HVAC,26817,15710,5977


Total series: 18


In [ ]:

# Le 10 serie interessate dal GBM e dall'Hybrid

target_series = [
    ("NE", "HVAC"),
    ("NE", "Plugs & Sockets"),
    ("NW", "HVAC"),
    ("NW", "Lighting"),
    ("NW", "Plugs & Sockets"),
    ("SE", "HVAC"),
    ("SE", "Lighting"),
    ("SE", "Plugs & Sockets"),
    ("SW", "Lighting"),
    ("SW", "Plugs & Sockets")
]

print("Total target series:", len(target_series))


selected = pd.MultiIndex.from_tuples(
    target_series, names=["zone", "load_type"]
)

# Contiamo le osservazioni per periodo
period_summary = (
    df.set_index(["zone", "load_type"])
      .loc[selected]
      .reset_index()
      .groupby(["zone", "load_type", "split"])
      .agg(
          total_rows=("zone", "size"),
          train_eligible=("include_train", "sum"),
          eval_eligible=("include_eval", "sum")
      )
      .reset_index()
)

display(period_summary)


Total target series: 10


,zone,load_type,split,total_rows,train_eligible,eval_eligible
0,NE,HVAC,test_2023,1416,0,1389
1,NE,HVAC,test_2025,1341,0,1287
2,NE,HVAC,train,18240,15169,0
3,NE,HVAC,unused,1417,0,0
4,NE,HVAC,validation,4416,0,4066
5,NE,Plugs & Sockets,test_2023,1416,0,1389
6,NE,Plugs & Sockets,test_2025,1341,0,1285
7,NE,Plugs & Sockets,train,18240,18084,0
8,NE,Plugs & Sockets,unused,1417,0,0
9,NE,Plugs & Sockets,validation,4416,0,4066


In [ ]:

# Verifica preliminare delle serie temporali

check = df[
    (df["zone"] == "NE") &
    (df["load_type"] == "Plugs & Sockets")
].copy()

check["timestamp_utc"] = pd.to_datetime(
    check["timestamp_utc"], utc=True
)

check = check.sort_values("timestamp_utc")

print("Rows:", len(check))
print("Duplicate timestamps:",
      check["timestamp_utc"].duplicated().sum())

time_diff = check["timestamp_utc"].diff()

print("\nTime intervals:")
print(time_diff.value_counts().head(10))

print("\nGaps longer than one hour:",
      (time_diff > pd.Timedelta(hours=1)).sum())


Rows: 26830
Duplicate timestamps: 0

Time intervals:
timestamp_utc
0 days 01:00:00      26828
901 days 19:00:00        1
Name: count, dtype: int64

Gaps longer than one hour: 1


In [ ]:

# Controllo: lag basato sulle righe vs lag temporale reale

check = check.copy()

# Metodo originale: 24 righe precedenti
check["lag_24_rows"] = (
    check["energy_wh_model"].shift(24)
)

# Timestamp corrispondente a 24 righe prima
check["previous_row_time"] = (
    check["timestamp_utc"].shift(24)
)

# Verifichiamo che siano passate esattamente 24 ore
check["valid_lag_24"] = (
    check["timestamp_utc"] -
    check["previous_row_time"]
    == pd.Timedelta(hours=24)
)

print("Rows with invalid 24-hour lag:",
      (~check["valid_lag_24"]).sum())

print("\nInvalid lag rows by period:")
print(
    check.loc[~check["valid_lag_24"], "split"]
    .value_counts()
)

print("\nFirst invalid rows:")
display(
    check.loc[
        ~check["valid_lag_24"],
        ["timestamp_utc", "split", "previous_row_time"]
    ].head(30)
)


Rows with invalid 24-hour lag: 48

Invalid lag rows by period:
split
unused       24
test_2025    24
Name: count, dtype: int64

First invalid rows:


,timestamp_utc,split,previous_row_time
26830,2020-04-02 23:00:00+00:00,unused,NaT
26831,2020-04-03 00:00:00+00:00,unused,NaT
26832,2020-04-03 01:00:00+00:00,unused,NaT
26833,2020-04-03 02:00:00+00:00,unused,NaT
26834,2020-04-03 03:00:00+00:00,unused,NaT
26835,2020-04-03 04:00:00+00:00,unused,NaT
26836,2020-04-03 05:00:00+00:00,unused,NaT
26837,2020-04-03 06:00:00+00:00,unused,NaT
26838,2020-04-03 07:00:00+00:00,unused,NaT
26839,2020-04-03 08:00:00+00:00,unused,NaT


In [ ]:

# STEP 29 - Row-based vs time-based lag audit
# SW and SE Plugs & Sockets

lag_results = []

for zone, load_type in [
    ("SW", "Plugs & Sockets"),
    ("SE", "Plugs & Sockets")
]:
    s = df[
        (df["zone"] == zone) &
        (df["load_type"] == load_type)
    ].copy()

    s["timestamp_utc"] = pd.to_datetime(
        s["timestamp_utc"], utc=True
    )

    s = s.sort_values("timestamp_utc")

    assert not s["timestamp_utc"].duplicated().any()

    # Original method: 24 previous rows
    s["previous_row_time"] = (
        s["timestamp_utc"].shift(24)
    )

    # Check whether this is exactly 24 hours earlier
    s["valid_row_lag"] = (
        s["timestamp_utc"] -
        s["previous_row_time"]
        == pd.Timedelta(hours=24)
    )

    # Exclude the first 24 rows with no row-based lag
    eligible = s[
        s["previous_row_time"].notna()
    ].copy()

    # Analyse the full series and the SEL period
    for period, subset in [
        ("All periods", eligible),
        ("SEL 2022", eligible[
            (eligible["timestamp_utc"] >=
             pd.Timestamp("2022-10-16", tz="UTC")) &
            (eligible["timestamp_utc"] <
             pd.Timestamp("2023-01-01", tz="UTC"))
        ])
    ]:
        total = len(subset)
        invalid = (~subset["valid_row_lag"]).sum()

        lag_results.append({
            "zone": zone,
            "load_type": load_type,
            "period": period,
            "eligible_rows": total,
            "invalid_row_lags": int(invalid),
            "invalid_pct": (
                100 * invalid / total
                if total else np.nan
            )
        })

lag_audit = pd.DataFrame(lag_results)

print("ROW-BASED LAG AUDIT")
display(lag_audit.round(2))


ROW-BASED LAG AUDIT


,zone,load_type,period,eligible_rows,invalid_row_lags,invalid_pct
0,SW,Plugs & Sockets,All periods,26806,24,0.09
1,SW,Plugs & Sockets,SEL 2022,1848,0,0.00
2,SE,Plugs & Sockets,All periods,26806,24,0.09
3,SE,Plugs & Sockets,SEL 2022,1848,0,0.00


In [ ]:

# Costruiamo il lag corretto usando i timestamp UTC

past = check[
    ["timestamp_utc", "energy_wh_model"]
].copy()

past["timestamp_utc"] = (
    past["timestamp_utc"] + pd.Timedelta(hours=24)
)

past = past.rename(
    columns={"energy_wh_model": "lag_24_correct"}
)

check = check.merge(
    past,
    on="timestamp_utc",
    how="left",
    validate="one_to_one"
)

print("Rows:", len(check))
print("Missing corrected lag:",
      check["lag_24_correct"].isna().sum())

print("\nRows where the original lag was invalid:")
display(
    check.loc[
        ~check["valid_lag_24"],
        ["timestamp_utc", "split",
         "lag_24_rows", "lag_24_correct"]
    ].tail(30)
)


Rows: 26830
Missing corrected lag: 519

Rows where the original lag was invalid:


,timestamp_utc,split,lag_24_rows,lag_24_correct
18,2020-04-03 17:00:00+00:00,unused,NaN,NaN
19,2020-04-03 18:00:00+00:00,unused,NaN,NaN
20,2020-04-03 19:00:00+00:00,unused,NaN,NaN
21,2020-04-03 20:00:00+00:00,unused,NaN,NaN
22,2020-04-03 21:00:00+00:00,unused,NaN,NaN
23,2020-04-03 22:00:00+00:00,unused,NaN,NaN
25489,2025-08-18 18:00:00+00:00,test_2025,188.8020,NaN
25490,2025-08-18 19:00:00+00:00,test_2025,188.4040,NaN
25491,2025-08-18 20:00:00+00:00,test_2025,187.7560,NaN
25492,2025-08-18 21:00:00+00:00,test_2025,186.0420,NaN


In [ ]:

# Verifichiamo le cause dei lag mancanti

previous_times = set(check["timestamp_utc"])

check["previous_hour_exists"] = (
    check["timestamp_utc"] - pd.Timedelta(hours=24)
).isin(previous_times)

missing = check[check["lag_24_correct"].isna()].copy()

print("Total missing lags:", len(missing))

print("Missing previous timestamp:",
      (~missing["previous_hour_exists"]).sum())

print("Previous timestamp exists, but value is missing:",
      missing["previous_hour_exists"].sum())

print("\nMissing lags by period:")
print(missing["split"].value_counts())


Total missing lags: 519
Missing previous timestamp: 48
Previous timestamp exists, but value is missing: 471

Missing lags by period:
split
validation    344
unused         53
train          49
test_2025      49
test_2023      24
Name: count, dtype: int64


In [ ]:

features_to_check = [
    "timestamp_utc", "timestamp_local",
    "energy_wh_model",
    "temp_c", "occupancy", "lum_lux",
    "dh_heat", "dh_cool",
    "hour_local", "dayofweek_local",
    "month_local", "is_weekend", "is_holiday",
    "include_train", "include_eval", "split"
]

print("Available features:")
for feature in features_to_check:
    print(
        feature,
        "OK" if feature in df.columns else "MISSING"
    )


Available features:
timestamp_utc OK
timestamp_local OK
energy_wh_model OK
temp_c OK
occupancy OK
lum_lux OK
dh_heat MISSING
dh_cool MISSING
hour_local OK
dayofweek_local OK
month_local OK
is_weekend OK
is_holiday OK
include_train OK
include_eval OK
split OK


In [ ]:

# Controllo delle variabili e dei periodi
columns = [
    "timestamp_utc", "zone", "load_type",
    "energy_wh_model", "temp_c",
    "occupancy", "lum_lux",
    "include_train", "include_eval", "split"
]

audit = df[columns].copy()

audit["timestamp_utc"] = pd.to_datetime(
    audit["timestamp_utc"], utc=True
)

print("Total rows:", len(audit))
print("Missing energy values:",
      audit["energy_wh_model"].isna().sum())

print("\nMissing sensor values:")
print(audit[["temp_c", "occupancy", "lum_lux"]].isna().sum())

print("\nPeriods:")
print(audit["split"].value_counts())


Total rows: 470680
Missing energy values: 66426

Missing sensor values:
temp_c       76451
occupancy    75917
lum_lux      75917
dtype: int64

Periods:
split
train         323298
validation     75072
unused         30805
test_2023      24072
test_2025      17433
Name: count, dtype: int64


In [ ]:

# Check that all ten target series are selected

print("Target series:", len(target_series))
print("Dataset available:", "df" in globals())

assert len(target_series) == 10

if "df" in globals():
    available = set(
        zip(df["zone"], df["load_type"])
    )

    missing = [
        series for series in target_series
        if series not in available
    ]

    print("Missing series:", missing)
    assert not missing

print("Check passed!")


Target series: 10
Dataset available: True
Missing series: []
Check passed!


In [ ]:

# Preparazione delle feature day-ahead
# Nessuna modifica al dataset originale


target_series = [
    ("NE", "HVAC"),
    ("NE", "Plugs & Sockets"),
    ("NW", "HVAC"),
    ("NW", "Lighting"),
    ("NW", "Plugs & Sockets"),
    ("SE", "HVAC"),
    ("SE", "Lighting"),
    ("SE", "Plugs & Sockets"),
    ("SW", "Lighting"),
    ("SW", "Plugs & Sockets")
]


parts = []

for zone, load in target_series:
    s = df[
        (df["zone"] == zone) &
        (df["load_type"] == load)
    ].copy()

    s["timestamp_utc"] = pd.to_datetime(
        s["timestamp_utc"], utc=True
    )

    s = s.sort_values("timestamp_utc")

    # Timestamp come indice: lag sulle ore reali
    s = s.set_index("timestamp_utc")

    def historical_lag(column, hours):
        past = s[column].copy()
        past.index = past.index + pd.Timedelta(hours=hours)
        return past.reindex(s.index)

    # Consumo disponibile 24 e 168 ore prima
    s["energy_lag_24"] = historical_lag(
        "energy_wh_model", 24
    )
    s["energy_lag_168"] = historical_lag(
        "energy_wh_model", 168
    )

    # Media delle 24 ore che terminano a t-24
    past_energy = s["energy_wh_model"].rolling(
        window=24, min_periods=24
    ).mean()

    past_energy.index = (
        past_energy.index + pd.Timedelta(hours=24)
    )

    s["roll_mean_24_safe"] = past_energy.reindex(s.index)

    # Sensori misurati almeno 24 ore prima
    for col in ["temp_c", "occupancy", "lum_lux"]:
        s[col + "_lag24"] = historical_lag(col, 24)

    # Heating/cooling degree-hours storici
    s["dh_heat_safe"] = (
        18 - s["temp_c_lag24"]
    ).clip(lower=0)

    s["dh_cool_safe"] = (
        s["temp_c_lag24"] - 24
    ).clip(lower=0)

    parts.append(s.reset_index())

safe_df = pd.concat(parts, ignore_index=True)

print("Rows:", len(safe_df))
print("Series:", safe_df.groupby(
    ["zone", "load_type"]
).ngroups)

print("\nMissing safe features:")
print(safe_df[
    ["energy_lag_24", "energy_lag_168",
     "roll_mean_24_safe", "temp_c_lag24",
     "occupancy_lag24", "lum_lux_lag24"]
].isna().sum())


Rows: 268287
Series: 10

Missing safe features:
energy_lag_24        21354
energy_lag_168       23720
roll_mean_24_safe    55936
temp_c_lag24         64942
occupancy_lag24      64673
lum_lux_lag24        64673
dtype: int64


In [ ]:

# Check corrected features for all 10 series

required_features = [
    "energy_lag_24",
    "energy_lag_168",
    "roll_mean_24_safe",
    "temp_c_lag24",
    "occupancy_lag24",
    "lum_lux_lag24",
    "dh_heat_safe",
    "dh_cool_safe"
]

missing_columns = [
    col for col in required_features
    if col not in safe_df.columns
]

assert not missing_columns, (
    f"Missing feature columns: {missing_columns}"
)

feature_check = (
    safe_df.groupby(["zone", "load_type"])
    [required_features]
    .count()
)

display(feature_check)

print("\nSeries checked:", len(feature_check))
assert len(feature_check) == 10

print("Feature verification PASSED")


energy_lag_24  energy_lag_168  roll_mean_24_safe  \
zone load_type                                                           
NE   HVAC                     26317           26029              25958   
     Plugs & Sockets          26311           26023              25918   
NW   HVAC                     24801           24513              24350   
     Lighting                 26313           26025              25908   
     Plugs & Sockets          26313           26025              25919   
SE   HVAC                     24614           24396              19331   
     Lighting                 24686           24463              19002   
     Plugs & Sockets          24624           24386              19290   
SW   Lighting                 21549           21428              12947   
     Plugs & Sockets          21405           21279              13728   

                      temp_c_lag24  occupancy_lag24  lum_lux_lag24  \
zone load_type                                                       
NE   HVAC                        0                0              0   
     Plugs & Sockets             0                0              0   
NW   HVAC                    24571            24571          24571   
     Lighting                26074            26074          26074   
     Plugs & Sockets         26074            26074          26074   
SE   HVAC                    25257            25257          25257   
     Lighting                25278            25278          25278   
     Plugs & Sockets         25298            25298          25298   
SW   Lighting                25439            25579          25579   
     Plugs & Sockets         25354            25483          25483   

                      dh_heat_safe  dh_cool_safe  
zone load_type                                    
NE   HVAC                        0             0  
     Plugs & Sockets             0             0  
NW   HVAC                    24571         24571  
     Lighting                26074         26074  
     Plugs & Sockets         26074         26074  
SE   HVAC                    25257         25257  
     Lighting                25278         25278  
     Plugs & Sockets         25298         25298  
SW   Lighting                25439         25439  
     Plugs & Sockets         25354         25354


Series checked: 10
Feature verification PASSED


In [ ]:

# Controllo temporale delle nuove feature

sample = safe_df[
    (safe_df["zone"] == "NE") &
    (safe_df["load_type"] == "Plugs & Sockets")
].copy()

sample = sample.sort_values("timestamp_utc")

# Controlliamo i primi valori disponibili
valid_sample = sample.dropna(
    subset=["energy_lag_24", "roll_mean_24_safe"]
)

display(
    valid_sample[
        ["timestamp_utc",
         "energy_wh_model",
         "energy_lag_24",
         "roll_mean_24_safe",
         "temp_c_lag24"]
    ].head(10)
)

print("Rows with both historical features:",
      len(valid_sample))

print("Missing corrected rolling mean:",
      sample["roll_mean_24_safe"].isna().sum())


,timestamp_utc,energy_wh_model,energy_lag_24,roll_mean_24_safe,temp_c_lag24
26877,2020-04-04 22:00:00+00:00,225.3155,224.8490,220.945521,NaN
26878,2020-04-04 23:00:00+00:00,226.9300,228.1960,221.244521,NaN
26879,2020-04-05 00:00:00+00:00,222.9090,228.5810,221.594542,NaN
26880,2020-04-05 01:00:00+00:00,217.8665,228.0505,221.957917,NaN
26881,2020-04-05 02:00:00+00:00,218.6315,224.0975,222.086667,NaN
26882,2020-04-05 03:00:00+00:00,217.6135,220.0515,222.057146,NaN
26883,2020-04-05 04:00:00+00:00,217.4070,218.6295,222.072833,NaN
26884,2020-04-05 05:00:00+00:00,220.9895,218.0785,222.060250,NaN
26885,2020-04-05 06:00:00+00:00,221.9305,217.6240,222.063792,NaN
26886,2020-04-05 07:00:00+00:00,219.6355,216.8010,221.982896,NaN


Rows with both historical features: 25918
Missing corrected rolling mean: 912


In [ ]:

import numpy as np
import pandas as pd

# Serie da verificare
s = safe_df[
    (safe_df["zone"] == "NE") &
    (safe_df["load_type"] == "Plugs & Sockets")
].copy()

s = s.sort_values("timestamp_utc")

# Recuperiamo il consumo esattamente 24 ore prima
historical = s[
    ["timestamp_utc", "energy_wh_model"]
].copy()

historical["timestamp_utc"] += pd.Timedelta(hours=24)

historical = historical.rename(
    columns={"energy_wh_model": "expected_lag24"}
)

audit_check = s.merge(
    historical,
    on="timestamp_utc",
    how="left",
    validate="one_to_one"
)

# Confrontiamo i valori calcolati
valid = audit_check[
    ["energy_lag_24", "expected_lag24"]
].dropna()

assert np.allclose(
    valid["energy_lag_24"],
    valid["expected_lag24"]
)

print("Historical lag verification: PASSED")
print("Rows verified:", len(valid))


Historical lag verification: PASSED
Rows verified: 26311


In [ ]:

# Correzione della rolling mean per tutte le 10 serie
# Utilizziamo timestamp reali, senza attraversare i gap

corrected_parts = []

for (zone, load), group in safe_df.groupby(
    ["zone", "load_type"], sort=False
):
    group = group.sort_values("timestamp_utc").copy()
    group = group.set_index("timestamp_utc")

    energy = group["energy_wh_model"]

    # Crea una griglia oraria regolare
    hourly = energy.asfreq("h")

    # Media delle 24 ore consecutive, fino a t-24
    rolling = hourly.rolling(
        window=24,
        min_periods=24
    ).mean()

    rolling.index = rolling.index + pd.Timedelta(hours=24)

    group["roll_mean_24_safe"] = rolling.reindex(
        group.index
    )

    corrected_parts.append(group.reset_index())

safe_df = pd.concat(corrected_parts, ignore_index=True)

print("Rows:", len(safe_df))
print("Series:", safe_df.groupby(
    ["zone", "load_type"]
).ngroups)

print("Missing corrected rolling means:",
      safe_df["roll_mean_24_safe"].isna().sum())


Rows: 268287
Series: 10
Missing corrected rolling means: 56083


In [ ]:

# Verifica indipendente della rolling mean

s = safe_df[
    (safe_df["zone"] == "NE") &
    (safe_df["load_type"] == "Plugs & Sockets")
].copy()

s["timestamp_utc"] = pd.to_datetime(
    s["timestamp_utc"], utc=True
)

energy_by_time = s.set_index(
    "timestamp_utc"
)["energy_wh_model"]

# Calcoliamo indipendentemente la media
# delle 24 ore da t-47 a t-24 inclusi

expected = []

for t in s["timestamp_utc"]:
    hours = pd.date_range(
        end=t - pd.Timedelta(hours=24),
        periods=24,
        freq="h"
    )

    values = energy_by_time.reindex(hours)

    expected.append(
        values.mean() if values.notna().all()
        else np.nan
    )

s["expected_rolling_mean"] = expected

# Confrontiamo sia valori sia NaN
assert np.allclose(
    s["roll_mean_24_safe"].to_numpy(dtype=float),
    s["expected_rolling_mean"].to_numpy(dtype=float),
    equal_nan=True
)

print("Rolling mean verification: PASSED")
print("Rows checked:", len(s))
print("Valid rolling means:",
      s["roll_mean_24_safe"].notna().sum())


Rolling mean verification: PASSED
Rows checked: 26830
Valid rolling means: 25895


In [ ]:

# Confrontiamo le due medie mobili

actual = s["roll_mean_24_safe"]
expected = s["expected_rolling_mean"]

different = ~np.isclose(
    actual,
    expected,
    equal_nan=True,
    rtol=1e-7,
    atol=1e-8
)

print("Total rows:", len(s))
print("Different rows:", different.sum())

print("\nDifferences by period:")
print(s.loc[different, "split"].value_counts())

print("\nFirst differences:")
display(
    s.loc[
        different,
        ["timestamp_utc", "split",
         "roll_mean_24_safe",
         "expected_rolling_mean"]
    ].head(15)
)


Total rows: 26830
Different rows: 0

Differences by period:
Series([], Name: count, dtype: int64)

First differences:


,timestamp_utc,split,roll_mean_24_safe,expected_rolling_mean


In [ ]:

import numpy as np
import pandas as pd

sensor_columns = ["temp_c", "occupancy", "lum_lux"]

for (zone, load), group in safe_df.groupby(
    ["zone", "load_type"]
):
    group = group.sort_values("timestamp_utc").copy()

    for sensor in sensor_columns:
        past = group[
            ["timestamp_utc", sensor]
        ].copy()

        past["timestamp_utc"] += pd.Timedelta(hours=24)

        past = past.rename(
            columns={sensor: "expected"}
        )

        test = group[
            ["timestamp_utc", sensor + "_lag24"]
        ].merge(
            past,
            on="timestamp_utc",
            how="left",
            validate="one_to_one"
        )

        assert np.allclose(
            test[sensor + "_lag24"].to_numpy(dtype=float),
            test["expected"].to_numpy(dtype=float),
            equal_nan=True
        ), f"Mismatch: {zone} {load} {sensor}"

    print(f"PASSED: {zone} – {load}")

print("\nAll historical sensor checks: PASSED")


PASSED: NE – HVAC
PASSED: NE – Plugs & Sockets
PASSED: NW – HVAC
PASSED: NW – Lighting
PASSED: NW – Plugs & Sockets
PASSED: SE – HVAC
PASSED: SE – Lighting
PASSED: SE – Plugs & Sockets
PASSED: SW – Lighting
PASSED: SW – Plugs & Sockets

All historical sensor checks: PASSED


In [ ]:

# Feature del GBM senza leakage individuato in T37
safe_features = [
    "hour_local",
    "dayofweek_local",
    "month_local",
    "is_weekend",
    "is_holiday",
    "temp_c_lag24",
    "dh_cool_safe",
    "dh_heat_safe",
    "occupancy_lag24",
    "lum_lux_lag24",
    "energy_lag_24",
    "energy_lag_168",
    "roll_mean_24_safe"
]

# Controlliamo che tutte le colonne esistano
assert all(c in safe_df.columns for c in safe_features)

# Manteniamo soltanto le righe con un consumo
# valido e le feature storiche essenziali

# Use the same mandatory-data filter as GBM training

model_data = safe_df.dropna(
    subset=[
        "energy_wh_model",
        "energy_lag_168",
        "roll_mean_24_safe"
    ]
).copy()

print("Rows before filtering:", len(safe_df))
print("Rows after filtering:", len(model_data))

print("\nRows by period:")
print(model_data["split"].value_counts())

print("\nMissing optional features:")
print(model_data[safe_features].isna().sum())


Rows before filtering: 268287
Rows after filtering: 199953

Rows by period:
split
train         142074
validation     30226
unused         10701
test_2023       9656
test_2025       7296
Name: count, dtype: int64

Missing optional features:
hour_local               0
dayofweek_local          0
month_local              0
is_weekend               0
is_holiday               0
temp_c_lag24         51970
dh_cool_safe         51970
dh_heat_safe         51970
occupancy_lag24      51952
lum_lux_lag24        51952
energy_lag_24            0
energy_lag_168           0
roll_mean_24_safe        0
dtype: int64


In [ ]:

# Controlliamo le osservazioni disponibili
# per training e valutazione

comparison_summary = (
    model_data.groupby(
        ["zone", "load_type", "split"]
    )
    .agg(
        total_rows=("energy_wh_model", "size"),
        train_rows=("include_train", "sum"),
        eval_rows=("include_eval", "sum")
    )
    .reset_index()
)

display(comparison_summary)

print("\nTotal training rows:",
      (model_data["include_train"] == 1).sum())

print("Total evaluation rows:",
      (model_data["include_eval"] == 1).sum())


,zone,load_type,split,total_rows,train_rows,eval_rows
0,NE,HVAC,test_2023,1321,0,1318
1,NE,HVAC,test_2025,1072,0,1049
2,NE,HVAC,train,17931,14946,0
3,NE,HVAC,unused,1141,0,0
4,NE,HVAC,validation,3873,0,3867
5,NE,Plugs & Sockets,test_2023,1321,0,1318
6,NE,Plugs & Sockets,test_2025,1031,0,1018
7,NE,Plugs & Sockets,train,17931,17824,0
8,NE,Plugs & Sockets,unused,1141,0,0
9,NE,Plugs & Sockets,validation,3873,0,3867



Total training rows: 132826
Total evaluation rows: 46007


In [ ]:

# Final check before training all 10 GBMs


# Corrected GBM features

corrected_features = [
    "hour_local",
    "dayofweek_local",
    "month_local",
    "is_weekend",
    "is_holiday",
    "temp_c_lag24",
    "dh_cool_safe",
    "dh_heat_safe",
    "occupancy_lag24",
    "lum_lux_lag24",
    "energy_lag_24",
    "energy_lag_168",
    "roll_mean_24_safe"
]

assert len(target_series) == 10
assert safe_df.groupby(["zone", "load_type"]).ngroups == 10

assert all(
    feature in safe_df.columns
    for feature in corrected_features
)

print("Ready to train 10 corrected GBM models.")
print("Features:", corrected_features)


Ready to train 10 corrected GBM models.
Features: ['hour_local', 'dayofweek_local', 'month_local', 'is_weekend', 'is_holiday', 'temp_c_lag24', 'dh_cool_safe', 'dh_heat_safe', 'occupancy_lag24', 'lum_lux_lag24', 'energy_lag_24', 'energy_lag_168', 'roll_mean_24_safe']


In [ ]:

from sklearn.ensemble import GradientBoostingRegressor
import pandas as pd
import numpy as np

corrected_predictions = []

for zone, load in target_series:

    s = safe_df[
        (safe_df["zone"] == zone) &
        (safe_df["load_type"] == load)
    ].copy()

    s = s.dropna(
        subset=[
            "energy_wh_model",
            "energy_lag_168",
            "roll_mean_24_safe"
        ]
    )

    train = s["include_train"] == 1
    evaluation = s["include_eval"] == 1

    model = GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=4,
        random_state=42
    )

    model.fit(
        s.loc[train, corrected_features].fillna(0),
        s.loc[train, "energy_wh_model"]
    )

    result = s.loc[
        evaluation,
        ["timestamp_utc", "zone", "load_type",
         "split", "energy_wh_model"]
    ].copy()

    result["pred_corrected"] = model.predict(
        s.loc[evaluation, corrected_features].fillna(0)
    )

    corrected_predictions.append(result)

    print(
        f"COMPLETED: {zone} - {load} | "
        f"Train: {train.sum()} | "
        f"Evaluation: {evaluation.sum()}"
    )

corrected_all = pd.concat(
    corrected_predictions,
    ignore_index=True
)

print("\nTotal corrected predictions:", len(corrected_all))
print("Series completed:", len(corrected_predictions))


COMPLETED: NE - HVAC | Train: 14946 | Evaluation: 6234
COMPLETED: NE - Plugs & Sockets | Train: 17824 | Evaluation: 6203
COMPLETED: NW - HVAC | Train: 16231 | Evaluation: 5466
COMPLETED: NW - Lighting | Train: 16469 | Evaluation: 6207
COMPLETED: NW - Plugs & Sockets | Train: 17333 | Evaluation: 6204
COMPLETED: SE - HVAC | Train: 12019 | Evaluation: 3603
COMPLETED: SE - Lighting | Train: 11417 | Evaluation: 3679
COMPLETED: SE - Plugs & Sockets | Train: 11148 | Evaluation: 4585
COMPLETED: SW - Lighting | Train: 7477 | Evaluation: 1762
COMPLETED: SW - Plugs & Sockets | Train: 7962 | Evaluation: 2064

Total corrected predictions: 46007
Series completed: 10


In [ ]:

# Verify updated GBM predictions

print("Corrected predictions available:",
      "corrected_all" in globals())

if "corrected_all" in globals():
    print("Rows:", len(corrected_all))
    print("Series:", corrected_all.groupby(
        ["zone", "load_type"]
    ).ngroups)

    print("\nAvailable splits:")
    print(corrected_all["split"].value_counts())


Corrected predictions available: True
Rows: 46007
Series: 10

Available splits:
split
validation    29524
test_2023      9394
test_2025      7089
Name: count, dtype: int64


In [ ]:

# Check NW HVAC performance on test 2023

nw = corrected_all[
    (corrected_all["zone"] == "NW") &
    (corrected_all["load_type"] == "HVAC") &
    (corrected_all["split"] == "test_2023")
].copy()

assert len(nw) > 0, "No NW HVAC test_2023 predictions"

y_true = nw["energy_wh_model"].to_numpy()
y_pred = nw["pred_corrected"].to_numpy()

rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
cv_rmse = 100 * rmse / np.mean(y_true)

print("NW HVAC - Test 2023")
print("Hours:", len(nw))
print(f"Corrected GBM CV(RMSE): {cv_rmse:.2f}%")
print("T37 reference: approximately 49.5%")


NW HVAC - Test 2023
Hours: 802
Corrected GBM CV(RMSE): 48.52%
T37 reference: approximately 49.5%


In [ ]:

# Export corrected GBM predictions - 10 series

from pathlib import Path
from google.colab import files

export_gbm = corrected_all[
    ["timestamp_utc", "zone", "load_type",
     "split", "energy_wh_model", "pred_corrected"]
].copy()

export_gbm = export_gbm.rename(columns={
    "energy_wh_model": "y_true",
    "pred_corrected": "y_pred"
})

export_gbm["model"] = "gbm_leakfree"

export_gbm = export_gbm[
    ["timestamp_utc", "zone", "load_type",
     "model", "split", "y_true", "y_pred"]
]

export_gbm["timestamp_utc"] = (
    pd.to_datetime(export_gbm["timestamp_utc"], utc=True)
    .dt.tz_convert(None)
)

output_file = Path("/content/GBM_Corrected_Predictions_10series.csv")

export_gbm.to_csv(
    output_file,
    index=False,
    date_format="%Y-%m-%d %H:%M:%S"
)

print("Saved:", output_file.name)
print("Rows:", len(export_gbm))
print("Series:", export_gbm.groupby(
    ["zone", "load_type"]
).ngroups)

files.download(str(output_file))


Saved: GBM_Corrected_Predictions_10series.csv
Rows: 46007
Series: 10


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

# Final CSV verification

from pathlib import Path

csv_path = Path("/content/GBM_Corrected_Predictions_10series.csv")

assert csv_path.exists(), "Corrected CSV not found"

check_csv = pd.read_csv(csv_path)

expected_columns = [
    "timestamp_utc", "zone", "load_type",
    "model", "split", "y_true", "y_pred"
]

assert check_csv.columns.tolist() == expected_columns
assert len(check_csv) == len(corrected_all)
assert check_csv.groupby(["zone", "load_type"]).ngroups == 10
assert check_csv["model"].eq("gbm_leakfree").all()
assert check_csv["timestamp_utc"].str.endswith("+00:00").sum() == 0

print("CSV verification PASSED")
print("Rows:", len(check_csv))
print("Series:", check_csv.groupby(["zone", "load_type"]).ngroups)
display(check_csv.head())


CSV verification PASSED
Rows: 46007
Series: 10


,timestamp_utc,zone,load_type,model,split,y_true,y_pred
0,2022-07-01 00:00:00,NE,HVAC,gbm_leakfree,validation,2.1355,2.486911
1,2022-07-01 01:00:00,NE,HVAC,gbm_leakfree,validation,2.1985,2.552966
2,2022-07-01 02:00:00,NE,HVAC,gbm_leakfree,validation,2.1560,2.548492
3,2022-07-01 03:00:00,NE,HVAC,gbm_leakfree,validation,2.1020,2.529090
4,2022-07-01 04:00:00,NE,HVAC,gbm_leakfree,validation,1.9560,2.533724


In [ ]:

# Final uniqueness and timestamp verification

check_csv["timestamp_utc"] = pd.to_datetime(
    check_csv["timestamp_utc"], utc=True
)

duplicates = check_csv.duplicated(
    ["timestamp_utc", "zone", "load_type"]
).sum()

missing_values = check_csv[
    ["timestamp_utc", "y_true", "y_pred"]
].isna().sum()

print("Duplicate prediction keys:", duplicates)
print("\nMissing values:")
print(missing_values)

assert duplicates == 0
assert missing_values.sum() == 0

print("\nFinal prediction integrity check PASSED")


Duplicate prediction keys: 0

Missing values:
timestamp_utc    0
y_true           0
y_pred           0
dtype: int64

Final prediction integrity check PASSED


In [ ]:

# Reproducibility information

import sys
import pandas as pd
import numpy as np
import sklearn

print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)

print("\nFinal model:")
print("GradientBoostingRegressor")
print("n_estimators=100")
print("learning_rate=0.1")
print("max_depth=4")
print("random_state=42")


Python: 3.13.18
Pandas: 2.2.3
NumPy: 2.4.6
Scikit-learn: 1.6.1

Final model:
GradientBoostingRegressor
n_estimators=100
learning_rate=0.1
max_depth=4
random_state=42


In [ ]:

# STEP 24 - Load official GBM predictions with corrected UTC timestamps

from google.colab import files
from pathlib import Path
import pandas as pd
import numpy as np

uploaded_official = files.upload()

official_file = next(
    (name for name in uploaded_official
     if name.startswith("All_Zones_GBM_predictions_utc")
     and name.endswith(".csv")),
    None
)

assert official_file is not None, (
    "Upload All_Zones_GBM_predictions_utc.csv"
)

official_gbm = pd.read_csv(official_file)

official_gbm["timestamp_utc"] = pd.to_datetime(
    official_gbm["timestamp_utc"],
    utc=True
)

print("Official GBM rows:", len(official_gbm))

print("Columns:")
print(official_gbm.columns.tolist())

print("\nOfficial series:")
display(
    official_gbm.groupby(["zone", "load_type"])
    .size().reset_index(name="rows")
)


Saving All_Zones_GBM_predictions_utc.csv to All_Zones_GBM_predictions_utc.csv
Official GBM rows: 48221
Columns:
['timestamp_utc', 'zone', 'load_type', 'model', 'split', 'y_true', 'y_pred']

Official series:


,zone,load_type,rows
0,NE,HVAC,6414
1,NE,Plugs & Sockets,6386
2,NW,HVAC,5626
3,NW,Lighting,6388
4,NW,Plugs & Sockets,6389
5,SE,HVAC,3910
6,SE,Lighting,4014
7,SE,Plugs & Sockets,4828
8,SW,Lighting,2017
9,SW,Plugs & Sockets,2249


In [ ]:

# STEP 26 - Official GBM vs corrected GBM on common SEL hours

official_compare = official_gbm[
    ["timestamp_utc", "zone", "load_type", "y_true", "y_pred"]
].copy().rename(columns={
    "y_true": "actual_official",
    "y_pred": "pred_original"
})

corrected_compare = export_gbm[
    ["timestamp_utc", "zone", "load_type", "y_true", "y_pred"]
].copy().rename(columns={
    "y_true": "actual_corrected",
    "y_pred": "pred_corrected"
})

# Ensure both timestamp columns refer to UTC instants
for data in [official_compare, corrected_compare]:
    data["timestamp_utc"] = pd.to_datetime(
        data["timestamp_utc"], utc=True
    )

keys = ["timestamp_utc", "zone", "load_type"]

assert not official_compare.duplicated(keys).any()
assert not corrected_compare.duplicated(keys).any()

matched_gbm = official_compare.merge(
    corrected_compare,
    on=keys,
    how="inner",
    validate="one_to_one"
)

# Compare actual consumption values
actual_diff = (
    matched_gbm["actual_official"] -
    matched_gbm["actual_corrected"]
).abs()

print("Matched observations:", len(matched_gbm))
print("Maximum actual difference:", actual_diff.max())
print("Actual mismatches > 0.01:", (actual_diff > 0.01).sum())

assert (actual_diff <= 0.01).all(), (
    "Official and corrected actual values do not match"
)

# Selection period: 16 October to 31 December 2022
sel_start = pd.Timestamp("2022-10-16", tz="UTC")
sel_end = pd.Timestamp("2023-01-01", tz="UTC")

sel_gbm = matched_gbm[
    (matched_gbm["timestamp_utc"] >= sel_start) &
    (matched_gbm["timestamp_utc"] < sel_end)
].copy()

def cv_rmse_percent(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    return 100 * np.sqrt(
        np.mean((actual - predicted) ** 2)
    ) / np.mean(actual)

comparison_rows = []

for (zone, load_type), g in sel_gbm.groupby(
    ["zone", "load_type"]
):
    original_cv = cv_rmse_percent(
        g["actual_official"], g["pred_original"]
    )
    corrected_cv = cv_rmse_percent(
        g["actual_corrected"], g["pred_corrected"]
    )

    comparison_rows.append({
        "zone": zone,
        "load_type": load_type,
        "common_hours": len(g),
        "original_GBM_CV": original_cv,
        "corrected_GBM_CV": corrected_cv,
        "difference_pp": corrected_cv - original_cv
    })

gbm_sel_comparison = pd.DataFrame(
    comparison_rows
).sort_values(["zone", "load_type"])

print("\nGBM COMPARISON - SEL 2022 (CV(RMSE) %)")
display(gbm_sel_comparison.round(2))

print("\nSeries compared:", len(gbm_sel_comparison))


Matched observations: 42693
Maximum actual difference: 0.0
Actual mismatches > 0.01: 0

GBM COMPARISON - SEL 2022 (CV(RMSE) %)


,zone,load_type,common_hours,original_GBM_CV,corrected_GBM_CV,difference_pp
0,NE,HVAC,1790,327.53,328.09,0.56
1,NE,Plugs & Sockets,1790,23.59,77.19,53.60
2,NW,HVAC,1583,31.34,33.89,2.55
3,NW,Lighting,1790,83.09,101.39,18.30
4,NW,Plugs & Sockets,1790,38.20,49.67,11.47
5,SE,HVAC,855,38.64,55.91,17.27
6,SE,Lighting,757,231.38,298.62,67.24
7,SE,Plugs & Sockets,1061,133.24,125.43,-7.82
8,SW,Lighting,341,333.04,461.86,128.83
9,SW,Plugs & Sockets,409,24.35,8.03,-16.32



Series compared: 10


In [ ]:

# STEP 27 - Export original vs corrected GBM comparison

from pathlib import Path
from google.colab import files

output_comparison = Path(
    "/content/GBM_Original_vs_Corrected_SEL_10series.csv"
)

gbm_sel_comparison.to_csv(
    output_comparison,
    index=False,
    float_format="%.4f"
)

print("Saved:", output_comparison.name)
print("Series:", len(gbm_sel_comparison))
print(
    "Original GBM better:",
    (gbm_sel_comparison["difference_pp"] > 0).sum()
)
print(
    "Corrected GBM better:",
    (gbm_sel_comparison["difference_pp"] < 0).sum()
)

files.download(str(output_comparison))


Saved: GBM_Original_vs_Corrected_SEL_10series.csv
Series: 10
Original GBM better: 8
Corrected GBM better: 2


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Final Results and Conclusions

## 1. Objective

This notebook extends the original T37 GBM leakage investigation to all ten energy consumption series covered by the original GBM model.

The objective is to assess the use of historical input features in a fixed 24-hour forecasting-horizon experiment, reducing the risk of using information unavailable at prediction time.

## 2. Methodology

The GBM models were retrained separately for each energy series using historical energy consumption, temperature, occupancy and luminosity features.

The original GBM hyperparameters were preserved:

- `n_estimators = 100`
- `learning_rate = 0.1`
- `max_depth = 4`
- `random_state = 42`

Historical lag features and rolling averages were constructed using timestamp-based calculations. Additional checks verified the rolling-window calculations and historical sensor alignment.

The corrected models were evaluated using the existing training and evaluation indicators, while accounting for unavailable historical observations.

## 3. Main Results

The analysis was extended from six to ten energy consumption series, including the four additional series:

- NW HVAC
- SE HVAC
- SW Lighting
- SW Plugs & Sockets

All ten corrected GBM models were successfully trained.

The experiment produced **46,007 predictions**, distributed as follows:

| Evaluation period | Predictions |
|---|---:|
| Validation | 29,524 |
| Test 2023 | 9,394 |
| Test 2025 | 7,089 |
| **Total** | **46,007** |

## 4. NW HVAC Verification

NW HVAC was used as a reference case to compare the new experiment with the original T37 investigation.

The corrected GBM achieved a **CV(RMSE) of 48.52%** on 802 observations in the 2023 test period.

The original T37 investigation reported approximately 49.5%.

Although the values are close, they are not identical. Differences in feature preparation or evaluation samples may contribute to the discrepancy. Therefore, this should be considered a consistency check rather than an exact reproduction.
## Original vs Corrected GBM — Selection Period

The original GBM and corrected GBM were compared across all ten energy consumption series during the T38 selection period (October 16 – December 31, 2022).

The comparison was restricted to matching UTC timestamps for each energy series. A total of 42,693 observations were matched across the full prediction outputs, and the actual consumption values were identical in both datasets.

On the common selection timestamps, the corrected GBM achieved higher CV(RMSE) in eight of the ten series and lower CV(RMSE) in two series.

The most notable increase occurred in NE Plugs & Sockets, where CV(RMSE) increased from 23.59% to 77.19% across 1,790 common selection hours.

By contrast, SW Plugs & Sockets improved from 24.35% to 8.03%, demonstrating that the impact was not uniform across energy series.

These findings suggest that the original GBM performance was sensitive to the feature configuration. However, differences cannot be attributed solely to data leakage because the corrected model also uses a modified feature set and potentially a different effective training sample.

The reduction in prediction coverage from 48,221 original predictions to 46,007 corrected predictions must also be considered when reconstructing the final T38 model selection.

**Output:** `GBM_Original_vs_Corrected_SEL_10series.csv`
## 5. Output Validation

The corrected predictions were exported to:

`GBM_Corrected_Predictions_10series.csv`

The file follows the standard prediction format:

`timestamp_utc, zone, load_type, model, split, y_true, y_pred`

The model identifier `gbm_leakfree` distinguishes these experimental forecasts from the original GBM predictions.

The final dataset passed the following integrity checks:

- Ten energy series represented
- No duplicate prediction keys
- No missing timestamps, actual values or predicted values
- Consistent UTC timestamp formatting

The Python and library versions used for the experiment were also recorded for reproducibility.

## 6. Limitations

The corrected features were designed for a fixed 24-hour forecasting horizon. Full operational availability of every feature at the issuance time of a day-ahead forecast has not yet been demonstrated.

Changes in feature availability and the effective training sample mean that performance differences cannot be attributed exclusively to leakage correction.

The original NW HVAC T37 result was approximated but not reproduced exactly.

Furthermore, this notebook does not establish the final winning forecasting models across all fourteen energy series. The Hybrid model and other candidates require separate evaluation.
### Row-Based Lag Verification

An additional audit was conducted for SW Plugs & Sockets and SE Plugs & Sockets to investigate whether row-based energy lags could explain the performance improvements observed after GBM correction.

Across the full dataset, only 24 out of 26,806 eligible observations (0.09%) showed a mismatch between a 24-row shift and an actual 24-hour time difference in each series.

During the 2022 selection period, no such mismatches were detected in either series.

Therefore, incorrect 24-row temporal alignment does not appear to explain the observed selection-period performance improvements. Other factors, including changes in the feature set, effective training sample and evaluation coverage, require further investigation.

The original-versus-corrected GBM comparison used only 409 common selection hours for SW Plugs & Sockets and 341 for SW Lighting. These reduced samples limit direct comparability with the original T38 results.
The underlying cause of the improvements in SW and SE Plugs & Sockets remains unidentified. Possible explanations include differences in the effective training sample and noisy or missing same-hour indoor sensor measurements used by the original GBM. These hypotheses have not yet been independently verified.
## 7. Next Steps

The corrected GBM predictions will be used in:

- **Notebook 03 – Hybrid Re-evaluation:** retraining the Hybrid model with corrected GBM predictions and implementing the original FIT, SEL and full-validation weight-estimation procedure.
- **Notebook 04 – Final Model Selection Audit:** re-evaluating the model-selection procedure and comparing all available candidates on consistent timestamps.

The two analyses will remain separate from this notebook to ensure traceability of the individual contributions.

**References:** `T37_check_gbm_leakage.py`, `GBM_forallzones.ipynb` and `01_GBM_Timezone_Audit.ipynb`.